In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WikimediaKafkaStreaming")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    )
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/15 12:10:03 WARN Utils: Your hostname, DESKTOP-FBRFLFL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/01/15 12:10:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/or/spark-streaming/wikimedia/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/or/.ivy2.5.2/cache
The jars for the packages stored in: /home/or/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-59ff4bbb-ca39-47c7-8cd2-3be6520f53fe;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in centr

In [2]:
df_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "wikimedia")
    .option("startingOffsets", "latest")
    .load()
)

df_raw.printSchema()


root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [3]:
from pyspark.sql.functions import col

df_text = df_raw.select(col("value").cast("string").alias("json"))

query = (
    df_text.writeStream
    .format("console")
    .option("truncate", "false")
    .start()
)

query


26/01/15 12:11:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cdd375be-5e0c-4ee2-8515-a7666ad2ee4e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/15 12:11:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+----+
|json|
+----+
+----+



In [4]:
query.stop()


26/01/15 12:12:43 WARN DAGScheduler: Failed to cancel job group a1f8f67d-2a46-4d5a-b83d-9f06a907b34d. Cannot find active jobs for it.
26/01/15 12:12:43 WARN DAGScheduler: Failed to cancel job group a1f8f67d-2a46-4d5a-b83d-9f06a907b34d. Cannot find active jobs for it.


In [5]:
df_raw2 = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "wikimedia")
    .option("startingOffsets", "earliest")
    .load()
)

df_text2 = df_raw2.select(col("value").cast("string").alias("json"))

query2 = (
    df_text2.writeStream
    .format("console")
    .option("truncate", "false")
    .start()
)

query2


26/01/15 12:12:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1207e221-a273-4068-870b-bef6ff4745cb. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/15 12:12:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------